# Tutorial 03 — Pretraining: data, objectives, scaling, and small-model training

[Lecture notes](../lectures/03_pretraining.md) · [Exercise sheet](../exercises/03_pretraining.md)

**40 minutes, CPU, no accounts or model downloads.** Requires Python 3.10+ and NumPy. These are mechanism experiments using authored or synthetic data, not frontier benchmark results. Restart the kernel and run all cells in order.

**Objective:** Train a neural character bigram model from scratch, print learning curves, and compare held-out loss with a uniform baseline.

Allocate 5 minutes to setup, 15 to the mechanism, 10 to interventions, and 10 to discussion. Record predictions before running. Complete the separate exercise sheet after class.


In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)


Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Build document-bounded training pairs

The small model has one learned next-character logit row per previous character. It exposes the full gradient without requiring a GPU.

Before running: predict the output or the invariant being checked.


In [2]:
train_docs=["the class starts today", "the lab opens today", "the class studies models", "the lab studies language"]
dev_docs=["the class opens today", "the lab starts today"]
vocab=["<unk>","<bos>","<eos>"]+sorted(set("".join(train_docs)))
ids={c:i for i,c in enumerate(vocab)}
def pairs(docs):
    pairs=[]
    for doc in docs:
        s=[1]+[ids.get(c,0) for c in doc]+[2]
        pairs.extend(zip(s[:-1],s[1:]))
    return np.array(pairs).T
x,y=pairs(train_docs); dx,dy=pairs(dev_docs)
W=rng.normal(0,.01,(len(vocab),len(vocab)))
def softmax(z):
    z=z-z.max(-1,keepdims=True); e=np.exp(z); return e/e.sum(-1,keepdims=True)
def loss(W,x,y): return -np.log(softmax(W[x])[np.arange(len(y)),y]).mean()
print("initial dev loss",loss(W,dx,dy))


initial dev loss 3.095091471214213


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 2. Train with an explicit gradient

Predict the gradient row sum. np.add.at accumulates repeated contexts correctly; ordinary advanced-index assignment would not.

Before running: predict the output or the invariant being checked.


In [3]:
history=[]
initial=loss(W,x,y)
for step in range(301):
    p=softmax(W[x]); delta=p.copy(); delta[np.arange(len(y)),y]-=1
    grad=np.zeros_like(W); np.add.at(grad,x,delta/len(y))
    assert np.allclose(grad.sum(axis=1),0,atol=1e-12)
    W-=2.0*grad
    if step%50==0:
        row=(step,float(loss(W,x,y)),float(loss(W,dx,dy)))
        history.append(row); print("step/train/dev",row)
assert loss(W,x,y)<initial


step/train/dev (0, 3.045231012170305, 3.0446233298431484)
step/train/dev (50, 1.6905940634511638, 1.6635518592424507)
step/train/dev (100, 1.356801146817496, 1.3318752777315355)
step/train/dev (150, 1.22394333858768, 1.2022335226263805)
step/train/dev (200, 1.1564515259084982, 1.1372884440034778)
step/train/dev (250, 1.1177126100133161, 1.1003185146807442)
step/train/dev (300, 1.0933783461892714, 1.077142523787638)


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 3. Finite-difference check and sample

A gradient check tests the derivation. Sampled text may still be poor: this model only remembers one previous character.

Before running: predict the output or the invariant being checked.


In [4]:
p=softmax(W[x]); p[np.arange(len(y)),y]-=1
G=np.zeros_like(W); np.add.at(G,x,p/len(y))
i,j=int(x[0]),int(y[0]); eps=1e-5
plus=W.copy(); minus=W.copy(); plus[i,j]+=eps; minus[i,j]-=eps
fd=(loss(plus,x,y)-loss(minus,x,y))/(2*eps)
assert np.isclose(fd,G[i,j],atol=1e-7)
cur=1; output=[]
for _ in range(100):
    cur=int(rng.choice(len(vocab),p=softmax(W[cur])))
    if cur==2: break
    output.append(vocab[cur])
print("sample:","".join(output))
print("FLOP estimate for 100M parameters / 2B tokens:",6*100_000_000*2_000_000_000)


sample: tmodie clage la
FLOP estimate for 100M parameters / 2B tokens: 1200000000000000000


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## Practical discussion

1. Why can training perplexity improve while useful performance gets worse?
2. How would duplicated test documents distort a scaling comparison?
3. Why is a randomly initialized bigram model not a small replica of a frontier LLM?


## Extension and submission

Run `extensions/train_tiny_decoder.py` for an actual randomly initialized causal Transformer trained on the same authored corpus. The tiny run demonstrates training, not useful language generation.

See the [extension guide](../extensions/README.md) before running model downloads. Extensions require additional dependencies and are not necessary for full exercise credit.

Submit the completed notebook with a 150–250-word interpretation and the separate exercise answers. Include a baseline, one controlled change, data provenance, seed, and an honest limitation. Never replace measured results with expected trends.
